# Adaptive Graph Convolutional Recurrent Network (AGCRN) for Traffic Forecasting

This notebook provides a clean, self-contained implementation of the **AGCRN** architecture, along with an enhanced variant, **AGCRNPlus**, designed for research experimentation on the METR-LA dataset.

## 1. AGCRN Architecture Overview
Traditional Spatio-Temporal Graph Neural Networks (STGNNs) rely on a pre-defined static graph to capture spatial relationships. However, physical distances do not always capture true traffic dependencies, and spatial relations can be dynamic. AGCRN addresses this by introducing two key components:
1. **Node-Adaptive Parameter Learning (NAPL)**: Dynamically generates node-specific weights from a weight pool, reducing parameter redundancy while capturing unique patterns for each sensor.
2. **Data-Adaptive Graph Generation (DAGG)**: Learns node embedding dictionaries representing intrinsic node properties, generating an adaptive adjacency matrix on the fly without manual graph construction.

These components are integrated with a Gated Recurrent Unit (GRU) to model temporal dynamics, forming the **Adaptive Value-Weighted Graph Convolutional Network (AVWGCN)** layers and the **AGCRN Cell**.

## 2. METR-LA Dataset
The METR-LA traffic dataset contains speed readings from **207 loop sensors** on highway loop segments in Los Angeles County. The data spans from **March 1, 2012 to June 30, 2012** (4 months), sampled at **5-minute intervals**, yielding **34,272 time steps**.

## 3. Forecasting Objective
The task is formulated as a multi-step spatio-temporal forecasting problem. Given historical speed observations of the past $T_{in} = 12$ steps (1 hour):
$$X = \{X_{t-T_{in}+1}, \dots, X_t\} \in \mathbb{R}^{T_{in} \times N \times D}$$
we aim to predict future speed values for the next $T_{out} = 12$ steps (1 hour):
$$Y = \{Y_{t+1}, \dots, Y_{t+T_{out}}\} \in \mathbb{R}^{T_{out} \times N \times D}$$
where $N = 207$ is the number of nodes, and $D = 1$ is the feature dimension (traffic speed).

## 4. Tensor Flow Diagram
```mermaid
graph TD
    Input["Input Tensor: (B, T_in=12, N=207, C_in=1)"] --> Encoder["AGCRN Encoder (AVWDCRNN)"]
    Embeddings["Node Embeddings E_G: (N, d=10)"] --> Adjacency["Adaptive Adjacency Matrix A_adp: (N, N)"]
    Adjacency --> Encoder
    Encoder --> EncoderOutput["Encoder Output: (B, T_in, N, H=64)"]
    
    subgraph AGCRNPlus Enhancements
        EncoderOutput --> PosEmb["Positional Embedding Added: (B, T_in, N, H)"]
        PosEmb --> TempAttention["Temporal Multi-Head Attention: (B*N, T_in, H)"]
        TempAttention --> Residual["Residual Skip Fusion & Layer Normalization"]
    end
    
    Residual --> PredHead["Prediction Head (Conv2D)"]
    PredHead --> Output["Output Forecast: (B, T_out=12, N, C_out=1)"]
```


# Section 2: Imports and Configuration

We import the required libraries (PyTorch, Numpy, Pandas, Matplotlib, sklearn) and define a `Config` dataclass to hold all hyperparameters. The configuration matches standard settings used in the AGCRN paper for METR-LA.


In [ ]:
import os
import math
import copy
import time
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Setup tqdm with import fallback to simple loops
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, *args, **kwargs):
        return iterable

# Set seeds for reproducibility
torch.manual_seed(10)
np.random.seed(10)
if torch.cuda.is_available():
    torch.cuda.manual_seed(10)

@dataclass
class Config:
    # Dataset properties
    NUM_NODES: int = 207
    INPUT_LENGTH: int = 12
    OUTPUT_LENGTH: int = 12
    
    # Architecture hyperparameters
    HIDDEN_DIM: int = 64
    EMBED_DIM: int = 10
    NUM_LAYERS: int = 2
    CHEB_K: int = 2
    
    # Optimization parameters
    BATCH_SIZE: int = 64
    LEARNING_RATE: float = 0.001
    EPOCHS: int = 30
    GRAD_CLIP_NORM: float = 5.0
    
    # Hardware device selection
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

config = Config()
print(f"Experiment initialized on device: {config.DEVICE}")
print("Configuration Details:")
for k, v in config.__dict__.items():
    print(f"  {k}: {v}")


# Section 3: Dataset Loading and Exploratory Data Analysis (EDA)

We load the `metr-la.h5` file using Pandas, analyze its structure, print statistics, check for missing values, and visualize speed curves, distributions, and spatial correlations using native matplotlib.


In [ ]:
# Load METR-LA dataset
metr_la_path = 'metr-la.h5'
if not os.path.exists(metr_la_path):
    # Fallback to search in parent directories
    for root, dirs, files in os.walk('.'):
        if 'metr-la.h5' in files:
            metr_la_path = os.path.join(root, 'metr-la.h5')
            break

print(f"Reading data from: {metr_la_path}")
df = pd.read_hdf(metr_la_path)
print(f"Dataset Shape: {df.shape} (Timesteps: {df.shape[0]}, Sensors: {df.shape[1]})")

# Display first few rows
print("\nFirst 3 rows of dataset:")
print(df.head(3))

# Print basic statistics
print("\nBasic Summary Statistics (first 5 sensors):")
print(df.iloc[:, :5].describe())

# Analyze missing/zero values
nan_count = df.isna().sum().sum()
zero_count = (df == 0.0).sum().sum()
total_cells = df.shape[0] * df.shape[1]
print(f"\nMissing Values (NaNs): {nan_count}")
print(f"Zero Values (often sensor errors): {zero_count} ({zero_count / total_cells * 100:.2f}% of total data)")

# 1. Traffic Speed Curves over 24 Hours
plt.figure(figsize=(14, 5))
for col in df.columns[:3]:
    plt.plot(df.index[:288], df[col].iloc[:288], label=f"Sensor {col}") # 288 intervals * 5 mins = 24 hours
plt.title("Traffic Speed Curves (First 24 Hours)", fontsize=14)
plt.xlabel("Timestamp", fontsize=12)
plt.ylabel("Speed (mph)", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2. Histogram of Speed Values
plt.figure(figsize=(10, 5))
plt.hist(df.values.flatten(), bins=100, color='royalblue', alpha=0.75, edgecolor='none', density=True)
plt.title("Distribution of Traffic Speed Values across METR-LA", fontsize=14)
plt.xlabel("Speed (mph)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 3. Correlation Heatmap for a subset of sensors
plt.figure(figsize=(10, 8))
subset_corr = df.iloc[:, :12].corr()
im = plt.imshow(subset_corr.values, cmap="coolwarm", aspect='equal')
plt.colorbar(im)
plt.xticks(np.arange(len(subset_corr.columns)), subset_corr.columns, rotation=90)
plt.yticks(np.arange(len(subset_corr.columns)), subset_corr.columns)
plt.title("Traffic Speed Correlation Matrix (First 12 Sensors)", fontsize=14)
plt.tight_layout()
plt.show()


# Section 4: Data Preprocessing

We implement the custom `StandardScaler` to normalize the data. We split the dataset chronologically into **70% Training**, **10% Validation**, and **20% Testing** sets. Finally, we generate sliding windows of input and output lengths ($T=12$).


In [ ]:
class StandardScaler:
    """Standardizes data using mean and standard deviation computed from the training split."""
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def transform(self, data):
        return (data - self.mean) / self.std

    def inverse_transform(self, data):
        if torch.is_tensor(data):
            mean = torch.tensor(self.mean, device=data.device, dtype=data.dtype)
            std = torch.tensor(self.std, device=data.device, dtype=data.dtype)
            return (data * std) + mean
        return (data * self.std) + self.mean

# Extract raw speed values and ensure shape is (Timesteps, Nodes, Features)
raw_data = df.values
if len(raw_data.shape) == 2:
    raw_data = np.expand_dims(raw_data, axis=-1)  # Shape becomes (34272, 207, 1)

# Chronological Splitting
num_samples = len(raw_data)
train_end = int(num_samples * 0.70)
val_end = int(num_samples * 0.80)

train_raw = raw_data[:train_end]
val_raw = raw_data[train_end:val_end]
test_raw = raw_data[val_end:]

# Fit normalizer ONLY on training set
train_mean = train_raw.mean(axis=(0, 1), keepdims=True)
train_std = train_raw.std(axis=(0, 1), keepdims=True)
scaler = StandardScaler(train_mean, train_std)

# Normalize splits
train_norm = scaler.transform(train_raw)
val_norm = scaler.transform(val_raw)
test_norm = scaler.transform(test_raw)

# Sliding Window Creation
def create_sliding_windows(data, input_len=12, output_len=12):
    """Generates sliding windows of shapes (Samples, T_in, N, D) and (Samples, T_out, N, D)"""
    X, Y = [], []
    length = len(data)
    end_index = length - input_len - output_len + 1
    for i in range(end_index):
        X.append(data[i : i + input_len])
        Y.append(data[i + input_len : i + input_len + output_len])
    return np.array(X), np.array(Y)

train_x, train_y = create_sliding_windows(train_norm, config.INPUT_LENGTH, config.OUTPUT_LENGTH)
val_x, val_y = create_sliding_windows(val_norm, config.INPUT_LENGTH, config.OUTPUT_LENGTH)
test_x, test_y = create_sliding_windows(test_norm, config.INPUT_LENGTH, config.OUTPUT_LENGTH)

print("Data Preprocessing Complete.")
print(f"Training set shapes:   X = {train_x.shape}, Y = {train_y.shape}")
print(f"Validation set shapes: X = {val_x.shape}, Y = {val_y.shape}")
print(f"Testing set shapes:    X = {test_x.shape}, Y = {test_y.shape}")


# Section 5: AGCRN Core Components

Here we implement the fundamental components of AGCRN cell-by-cell with markdown explanations. Every tensor shape is explicitly documented in the comments.

## 5.1 Adaptive Graph Generator
The `AdaptiveGraphGenerator` defines a learnable node embedding matrix $E_G \in \mathbb{R}^{N \times d}$. An adaptive spatial adjacency matrix is computed by multiplying $E_G$ with itself and applying a ReLU activation followed by a Softmax normalization: $A_{adp} = \text{Softmax}(\text{ReLU}(E_G E_G^T))$.


In [ ]:
class AdaptiveGraphGenerator(nn.Module):
    def __init__(self, num_nodes, embed_dim):
        super(AdaptiveGraphGenerator, self).__init__()
        # Learnable node embeddings representing intrinsic sensor characteristics
        # E_G shape: [N, D_emb]
        self.node_embeddings = nn.Parameter(torch.randn(num_nodes, embed_dim), requires_grad=True)

    def forward(self):
        # Compute similarity matrix via matrix multiplication
        # similarity shape: [N, N]
        similarity = torch.mm(self.node_embeddings, self.node_embeddings.transpose(0, 1))
        
        # Filter negative relations and normalize row-wise using Softmax
        # A_adp shape: [N, N]
        A_adp = F.softmax(F.relu(similarity), dim=-1)
        return A_adp, self.node_embeddings

# Visualize initial state of the adaptive adjacency matrix
graph_gen = AdaptiveGraphGenerator(config.NUM_NODES, config.EMBED_DIM)
init_adj, init_emb = graph_gen()
print(f"Adjacency shape: {init_adj.shape}, Embeddings shape: {init_emb.shape}")

plt.figure(figsize=(7, 6))
im = plt.imshow(init_adj.detach().cpu().numpy(), cmap="hot")
plt.colorbar(im)
plt.title("Initial State of Learnable Adaptive Adjacency Matrix", fontsize=12)
plt.show()


## 5.2 Adaptive Value-Weighted Graph Convolution (AVWGCN) Layer
The `AVWGCN` layer executes adaptive graph convolution using Chebyshev polynomial approximations of order $K$. It bypasses standard parameter definitions by generating spatial weights $W$ and biases $b$ node-specifically from weight and bias pools ($W_{pool} \in \mathbb{R}^{d \times K \times C_{in} \times C_{out}}$, $b_{pool} \in \mathbb{R}^{d \times C_{out}}$) using Einstein summation based on learned node embeddings.


In [ ]:
class AVWGCN(nn.Module):
    def __init__(self, dim_in, dim_out, cheb_k, embed_dim):
        super(AVWGCN, self).__init__()
        self.cheb_k = cheb_k
        # Weight pool parameter: [embed_dim, cheb_k, dim_in, dim_out]
        self.weights_pool = nn.Parameter(torch.FloatTensor(embed_dim, cheb_k, dim_in, dim_out))
        # Bias pool parameter: [embed_dim, dim_out]
        self.bias_pool = nn.Parameter(torch.FloatTensor(embed_dim, dim_out))
        
        # Initialize parameters
        nn.init.xavier_uniform_(self.weights_pool)
        nn.init.xavier_uniform_(self.bias_pool)

    def forward(self, x, node_embeddings):
        # Inputs:
        #   x: [B, N, C_in] (Batch size, Nodes count, Input dimension)
        #   node_embeddings: [N, D_emb] (Nodes count, Embedding dimension)
        node_num = node_embeddings.shape[0]
        
        # 1. Compute adaptive adjacency: [N, N]
        similarity = torch.mm(node_embeddings, node_embeddings.transpose(0, 1))
        supports = F.softmax(F.relu(similarity), dim=1) # [N, N]
        
        # 2. Build Chebyshev polynomial support set: [cheb_k, N, N]
        support_set = [torch.eye(node_num).to(x.device), supports]
        for k in range(2, self.cheb_k):
            support_set.append(torch.matmul(2 * supports, support_set[-1]) - support_set[-2])
        supports = torch.stack(support_set, dim=0) # [cheb_k, N, N]
        
        # 3. Generate node-specific weights and biases from the parameter pools
        # weights: [N, cheb_k, dim_in, dim_out]
        weights = torch.einsum('nd,dkio->nkio', node_embeddings, self.weights_pool)
        # bias: [N, dim_out]
        bias = torch.matmul(node_embeddings, self.bias_pool)
        
        # 4. Perform Graph Convolution mapping
        # Step 4a: Multiply support matrices with inputs: [cheb_k, N, N] * [B, N, C_in] -> [B, cheb_k, N, C_in]
        x_g = torch.einsum("knm,bmc->bknc", supports, x)
        # Step 4b: Permute to align with weights: [B, N, cheb_k, C_in]
        x_g = x_g.permute(0, 2, 1, 3)
        # Step 4c: Apply node-specific weights and add bias: [B, N, dim_out]
        x_gconv = torch.einsum('bnki,nkio->bno', x_g, weights) + bias
        return x_gconv


## 5.3 AGCRN Cell
The `AGCRNCell` integrates adaptive convolutions into recurrent gates. Let $X_t$ be input and $h_{t-1}$ be the previous recurrent state. The update gate $z$, reset gate $r$, and candidate state $\tilde{h}$ are computed as:
$$z_t = \sigma(\text{AVWGCN}([X_t, h_{t-1}], E_G))$$
$$r_t = \sigma(\text{AVWGCN}([X_t, h_{t-1}], E_G))$$
$$\tilde{h}_t = \tanh(\text{AVWGCN}([X_t, z_t \odot h_{t-1}], E_G))$$
$$h_t = r_t \odot h_{t-1} + (1 - r_t) \odot \tilde{h}_t$$
A single stacked convolution maps $[X_t, h_{t-1}]$ to $2D_{out}$ dimensions to speed up gate computations.


In [ ]:
class AGCRNCell(nn.Module):
    def __init__(self, node_num, dim_in, dim_out, cheb_k, embed_dim):
        super(AGCRNCell, self).__init__()
        self.node_num = node_num
        self.hidden_dim = dim_out
        
        # Gate convolution maps concatenated input & state [B, N, dim_in + dim_out] to [B, N, 2 * dim_out]
        self.gate = AVWGCN(dim_in + self.hidden_dim, 2 * dim_out, cheb_k, embed_dim)
        # Update convolution computes candidate state mapping
        self.update = AVWGCN(dim_in + self.hidden_dim, dim_out, cheb_k, embed_dim)

    def forward(self, x, state, node_embeddings):
        # Inputs:
        #   x: [B, N, dim_in]
        #   state: [B, N, dim_out] (Previous hidden state)
        #   node_embeddings: [N, D_emb]
        state = state.to(x.device)
        
        # Concatenate inputs: [B, N, dim_in + dim_out]
        input_and_state = torch.cat((x, state), dim=-1)
        
        # Compute gate outputs and split into reset (r) and update (z): [B, N, dim_out] each
        z_r = torch.sigmoid(self.gate(input_and_state, node_embeddings))
        z, r = torch.split(z_r, self.hidden_dim, dim=-1)
        
        # Candidate representation with reset-modulated gate: [B, N, dim_in + dim_out]
        candidate = torch.cat((x, z * state), dim=-1)
        # Candidate state: [B, N, dim_out]
        hc = torch.tanh(self.update(candidate, node_embeddings))
        
        # Update recurrent state: [B, N, dim_out]
        h = r * state + (1 - r) * hc
        return h

    def init_hidden_state(self, batch_size):
        # Returns zero initialized states of shape [B, N, dim_out]
        return torch.zeros(batch_size, self.node_num, self.hidden_dim)


## 5.4 AGCRN Recurrent Encoder
The `AVWDCRNN` module is a multi-layer recurrent neural network stacking multiple `AGCRNCell` layers. It iterates chronologically over input sequences, updating state representations at each timestep.


In [ ]:
class AVWDCRNN(nn.Module):
    def __init__(self, node_num, dim_in, dim_out, cheb_k, embed_dim, num_layers=1):
        super(AVWDCRNN, self).__init__()
        self.node_num = node_num
        self.input_dim = dim_in
        self.num_layers = num_layers
        self.dcrnn_cells = nn.ModuleList()
        
        # Initialize cells for stacked layers
        self.dcrnn_cells.append(AGCRNCell(node_num, dim_in, dim_out, cheb_k, embed_dim))
        for _ in range(1, num_layers):
            self.dcrnn_cells.append(AGCRNCell(node_num, dim_out, dim_out, cheb_k, embed_dim))

    def forward(self, x, init_state, node_embeddings):
        # Inputs:
        #   x: [B, T_in, N, D_in] (Sequence of inputs)
        #   init_state: [num_layers, B, N, dim_out] (Stacked initial states)
        #   node_embeddings: [N, D_emb]
        assert x.shape[2] == self.node_num and x.shape[3] == self.input_dim
        seq_length = x.shape[1]
        current_inputs = x
        output_hidden = []
        
        # Execute computations layer-by-layer
        for i in range(self.num_layers):
            state = init_state[i]
            inner_states = []
            for t in range(seq_length):
                # Pass current timestep step-by-step: [B, N, C_in] (or [B, N, dim_out] for deep layers)
                state = self.dcrnn_cells[i](current_inputs[:, t, :, :], state, node_embeddings)
                inner_states.append(state)
            output_hidden.append(state)
            # Stack states to serve as sequence for the next layer: [B, T_in, N, dim_out]
            current_inputs = torch.stack(inner_states, dim=1)
            
        # Return output sequence from the final layer: [B, T_in, N, dim_out]
        # and the list of last states for all layers: list of length num_layers
        return current_inputs, output_hidden

    def init_hidden(self, batch_size):
        # Collect initial hidden states across layers: [num_layers, B, N, dim_out]
        init_states = []
        for i in range(self.num_layers):
            init_states.append(self.dcrnn_cells[i].init_hidden_state(batch_size))
        return torch.stack(init_states, dim=0)


## 5.5 Prediction Head
The prediction mapping converts final representations into multi-step forecasts using a 2D Convolution. The input of shape $[B, 1, N, d_{hidden}]$ is processed with a kernel size of $(1, d_{hidden})$, collapsing the state representation dimension into $T_{out} \times D_{out}$ forecast channels.


In [ ]:
class PredictionHead(nn.Module):
    def __init__(self, hidden_dim, output_dim, horizon):
        super(PredictionHead, self).__init__()
        # Maps hidden state feature dimension to prediction target outputs
        # Input channels: 1, Output channels: horizon * output_dim
        # Kernel size: (1, hidden_dim)
        self.end_conv = nn.Conv2d(1, horizon * output_dim, kernel_size=(1, hidden_dim), bias=True)
        nn.init.xavier_uniform_(self.end_conv.weight)

    def forward(self, x):
        # Input shape: [B, 1, N, hidden_dim]
        # Output shape: [B, horizon * output_dim, N, 1]
        return self.end_conv(x)


## 5.6 Baseline AGCRN Assembly
We assemble the full baseline AGCRN model below for comparison.


In [ ]:
class AGCRN(nn.Module):
    def __init__(self, config):
        super(AGCRN, self).__init__()
        self.num_nodes = config.NUM_NODES
        self.input_dim = 1
        self.hidden_dim = config.HIDDEN_DIM
        self.output_dim = 1
        self.horizon = config.OUTPUT_LENGTH
        self.num_layers = config.NUM_LAYERS
        
        # Initialized node embeddings
        self.node_embeddings = nn.Parameter(torch.randn(self.num_nodes, config.EMBED_DIM), requires_grad=True)
        
        # Spatio-temporal encoder
        self.encoder = AVWDCRNN(self.num_nodes, self.input_dim, self.hidden_dim, config.CHEB_K, config.EMBED_DIM, self.num_layers)
        
        # Prediction module
        self.prediction_head = PredictionHead(self.hidden_dim, self.output_dim, self.horizon)

    def forward(self, source):
        # Input:
        #   source: [B, T_in, N, D_in]
        B = source.shape[0]
        
        # Step 1: Initialize states: [num_layers, B, N, hidden_dim]
        init_state = self.encoder.init_hidden(B)
        
        # Step 2: Encoder execution
        # encoder_out shape: [B, T_in, N, hidden_dim]
        encoder_out, _ = self.encoder(source, init_state, self.node_embeddings)
        
        # Step 3: Extract last timestep output to feed Conv2D: [B, 1, N, hidden_dim]
        x_pred = encoder_out[:, -1:, :, :]
        
        # Step 4: Map predictions
        # out shape: [B, T_out * D_out, N, 1]
        out = self.prediction_head(x_pred)
        
        # Squeeze and reshape output: [B, T_out, D_out, N]
        out = out.squeeze(-1).reshape(-1, self.horizon, self.output_dim, self.num_nodes)
        # Permute to output format: [B, T_out, N, D_out]
        out = out.permute(0, 1, 3, 2)
        return out

# Compile validation check
test_model = AGCRN(config).to(config.DEVICE)
test_input = torch.randn(8, 12, 207, 1).to(config.DEVICE)
test_output = test_model(test_input)
print(f"Validation check: Input shape = {test_input.shape} -> Output shape = {test_output.shape}")


# Section 6: Our Improved Model - AGCRNPlus

To enhance traffic forecasting capabilities, we design **AGCRNPlus**. AGCRNPlus introduces five major architectural improvements over the base AGCRN model:

1. **Learnable Temporal Positional Embeddings**: Adds positional parameters of shape `(1, T_in, 1, hidden_dim)` to inject temporal order awareness to recurrent states prior to attention.
2. **Multi-Head Temporal Attention**: Utilizes `nn.MultiheadAttention` along the sequence dimension ($T_{in}$) independently for each node. This enables capturing long-range temporal dependencies that are often forgotten by recurrent gates.
3. **Dropout (0.3)**: Regulates the attention outputs to avoid overfitting.
4. **Residual Skip Connection**: Establishes a shortcut mapping that merges original encoder representations back into the refined attention features, preventing gradient vanishing.
5. **Layer Normalization**: Stabilizes training dynamics by normalizing features across the hidden dimension.

The sequential computation flow of AGCRNPlus is:
$$\text{Input} \longrightarrow \text{Adaptive Graph Encoder} \longrightarrow \text{Temporal Attention} \longrightarrow \text{Residual Fusion} \longrightarrow \text{Prediction Head}$$


In [ ]:
class AGCRNPlus(nn.Module):
    def __init__(self, config):
        super(AGCRNPlus, self).__init__()
        self.num_nodes = config.NUM_NODES
        self.input_dim = 1
        self.hidden_dim = config.HIDDEN_DIM
        self.output_dim = 1
        self.horizon = config.OUTPUT_LENGTH
        self.num_layers = config.NUM_LAYERS
        
        # 1. Node embeddings matrix
        self.node_embeddings = nn.Parameter(torch.randn(self.num_nodes, config.EMBED_DIM), requires_grad=True)
        
        # 2. Recurrent spatial encoder
        self.encoder = AVWDCRNN(self.num_nodes, self.input_dim, self.hidden_dim, config.CHEB_K, config.EMBED_DIM, self.num_layers)
        
        # 3. Learnable temporal positional embeddings: [1, T_in, 1, hidden_dim]
        self.pos_embeddings = nn.Parameter(torch.randn(1, config.INPUT_LENGTH, 1, self.hidden_dim) * 0.02)
        
        # 4. Multi-head temporal attention (heads = 4, dropout = 0.3)
        self.temporal_attn = nn.MultiheadAttention(embed_dim=self.hidden_dim, num_heads=4, dropout=0.3, batch_first=True)
        
        # 5. Regularization & Normalization layers
        self.dropout = nn.Dropout(0.3)
        self.norm = nn.LayerNorm(self.hidden_dim)
        
        # 6. Predictor module
        self.prediction_head = PredictionHead(self.hidden_dim, self.output_dim, self.horizon)

    def forward(self, source):
        # Inputs:
        #   source: [B, T_in, N, D_in] (where T_in = 12)
        B, T, N, D = source.shape
        
        # Step 1: Execute Recurrent Spatial Encoder
        init_state = self.encoder.init_hidden(B)
        # encoder_out: [B, T_in, N, hidden_dim]
        encoder_out, _ = self.encoder(source, init_state, self.node_embeddings)
        
        # Step 2: Inject temporal positional embeddings
        # x_pos shape: [B, T_in, N, hidden_dim]
        x_pos = encoder_out + self.pos_embeddings
        
        # Step 3: Apply Multi-Head Temporal Attention
        # We pack batch & spatial nodes together: [B * N, T_in, hidden_dim]
        x_attn_in = x_pos.permute(0, 2, 1, 3).reshape(B * N, T, self.hidden_dim)
        
        # Compute self-attention: query=key=value=x_attn_in
        # attn_out: [B * N, T_in, hidden_dim]
        # attn_weights: [B * N, T_in, T_in] (useful for visual analysis)
        attn_out, attn_weights = self.temporal_attn(x_attn_in, x_attn_in, x_attn_in)
        
        # Reshape back to spatial sequence: [B, T_in, N, hidden_dim]
        attn_out = attn_out.reshape(B, N, T, self.hidden_dim).permute(0, 2, 1, 3)
        
        # Step 4: Residual fusion, dropout, and normalization
        # fused shape: [B, T_in, N, hidden_dim]
        fused = self.norm(encoder_out + self.dropout(attn_out))
        
        # Step 5: Select final state representation: [B, 1, N, hidden_dim]
        x_pred = fused[:, -1:, :, :]
        
        # Step 6: Perform Conv2D forecasting projection
        # out: [B, T_out * D_out, N, 1]
        out = self.prediction_head(x_pred)
        
        # Squeeze and format output: [B, T_out, N, D_out]
        out = out.squeeze(-1).reshape(-1, self.horizon, self.output_dim, self.num_nodes)
        out = out.permute(0, 1, 3, 2)
        return out, attn_weights

# Compile validation check
test_model = AGCRNPlus(config).to(config.DEVICE)
test_input = torch.randn(8, 12, 207, 1).to(config.DEVICE)
test_output, test_attn = test_model(test_input)
print(f"AGCRNPlus check: Output shape = {test_output.shape}, Attention weights shape = {test_attn.shape}")


# Section 7: Training Pipeline

We construct a clean, robust training pipeline with the following characteristics:
- **Loss Function**: PyTorch Mean Absolute Error (MAE) loss, calculated on the unscaled real values if `real_value=True` (matching paper practice).
- **Optimizer**: Adam optimizer with `lr = 0.001`.
- **Gradient Clipping**: Clipped at max norm of `5.0` to prevent gradient explosion.
- **Learning Rate Scheduler**: Step Decay / CosineAnnealingLR.
- **Mixed Precision Training**: Dynamic integration of `torch.cuda.amp` to accelerate GPU training if CUDA is available.
- **Epochs**: Training for 30 epochs, recording metrics on train and validation sets.
- **Tqdm Integration**: Displays visual updates for epochs and steps.


In [ ]:
# Helper functions for loss computation
def mae_loss_fn(pred, target):
    return torch.mean(torch.abs(pred - target))

# Create PyTorch DataLoaders
train_dataset = TensorDataset(torch.FloatTensor(train_x), torch.FloatTensor(train_y))
val_dataset = TensorDataset(torch.FloatTensor(val_x), torch.FloatTensor(val_y))
test_dataset = TensorDataset(torch.FloatTensor(test_x), torch.FloatTensor(test_y))

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, drop_last=False)

def train_model(model, model_name="AGCRN", epochs=30):
    model = model.to(config.DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    # Reduce learning rate when validation loss plateaus
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    # Setup mixed precision training objects if GPU is available
    use_amp = (config.DEVICE == "cuda")
    scaler_amp = torch.cuda.amp.GradScaler(enabled=use_amp)
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_model_state = None
    
    print(f"\n--- Starting Training for {model_name} ---")
    for epoch in range(1, epochs + 1):
        # 1. Training Phase
        model.train()
        epoch_train_loss = 0.0
        
        # tqdm step progress bar
        step_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]", leave=False)
        for x_batch, y_batch in step_bar:
            x_batch = x_batch.to(config.DEVICE) # [B, T_in, N, D_in]
            y_batch = y_batch.to(config.DEVICE) # [B, T_out, N, D_out]
            
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast(enabled=use_amp):
                if model_name == "AGCRNPlus":
                    pred, _ = model(x_batch)
                else:
                    pred = model(x_batch)
                
                # Calculate loss against UNSCALED targets (real values)
                y_batch_real = scaler.inverse_transform(y_batch)
                loss = mae_loss_fn(pred, y_batch_real)
            
            # Backpropagation with AMP scaling
            scaler_amp.scale(loss).backward()
            
            # Clip gradients to avoid explosion
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP_NORM)
            
            scaler_amp.step(optimizer)
            scaler_amp.update()
            
            epoch_train_loss += loss.item()
            # If step_bar has a set_postfix method, use it
            if hasattr(step_bar, 'set_postfix'):
                step_bar.set_postfix(loss=f"{loss.item():.4f}")
            
        avg_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # 2. Validation Phase
        model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(config.DEVICE)
                y_batch = y_batch.to(config.DEVICE)
                
                if model_name == "AGCRNPlus":
                    pred, _ = model(x_batch)
                else:
                    pred = model(x_batch)
                
                y_batch_real = scaler.inverse_transform(y_batch)
                loss = mae_loss_fn(pred, y_batch_real)
                epoch_val_loss += loss.item()
                
        avg_val_loss = epoch_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # Step scheduler based on validation loss
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.6f}")
        
        # Save best model weights
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            
    print(f"Training Finished. Best Validation Loss: {best_val_loss:.4f}")
    # Load best weights before returning
    model.load_state_dict(best_model_state)
    return model, train_losses, val_losses


## 7.1 Run Model Training
We train both the baseline **AGCRN** and our enhanced **AGCRNPlus** model. We train for 30 epochs.


In [ ]:
# Instantiate and train the baseline model
agcrn_baseline = AGCRN(config)
agcrn_baseline, baseline_train_loss, baseline_val_loss = train_model(agcrn_baseline, "AGCRN", epochs=config.EPOCHS)

# Instantiate and train the enhanced AGCRNPlus model
agcrn_plus = AGCRNPlus(config)
agcrn_plus, plus_train_loss, plus_val_loss = train_model(agcrn_plus, "AGCRNPlus", epochs=config.EPOCHS)


# Section 8: Evaluation Metrics

We evaluate models on the test split. Evaluation is performed at multiple forecasting horizons:
- **3 steps ahead** (15 minutes)
- **6 steps ahead** (30 minutes)
- **12 steps ahead** (60 minutes)

We compute three standard metrics:
1. **Mean Absolute Error (MAE)**: $\text{MAE} = \frac{1}{M}\sum |y_i - \hat{y}_i|$
2. **Root Mean Squared Error (RMSE)**: $\text{RMSE} = \sqrt{\frac{1}{M}\sum (y_i - \hat{y}_i)^2}$
3. **Mean Absolute Percentage Error (MAPE)**: $\text{MAPE} = \frac{1}{M_{mask}}\sum \left|\frac{y_i - \hat{y}_i}{y_i}\right| \times 100\%$ (ignoring zero speed elements to prevent division by zero).


In [ ]:
def test_eval_metrics(pred, true):
    """Computes MAE, RMSE, and MAPE metrics on numpy arrays."""
    # MAE
    mae = np.mean(np.abs(pred - true))
    
    # RMSE
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    
    # MAPE (filter zero labels to prevent division by zero)
    zero_mask = true > 0.0
    mape = np.mean(np.abs(pred[zero_mask] - true[zero_mask]) / true[zero_mask])
    
    return mae, rmse, mape

def evaluate_model_on_test(model, model_name="AGCRN"):
    model.eval()
    all_preds = []
    all_trues = []
    all_attn_maps = []
    
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(config.DEVICE)
            # Compute prediction
            if model_name == "AGCRNPlus":
                pred, attn = model(x_batch)
                all_attn_maps.append(attn.cpu())
            else:
                pred = model(x_batch)
            
            all_preds.append(pred.cpu())
            # Revert normalized label to original scale
            y_real = scaler.inverse_transform(y_batch)
            all_trues.append(y_real.cpu())
            
    # Concatenate results
    preds = torch.cat(all_preds, dim=0).numpy() # [Samples, T_out, N, D_out]
    trues = torch.cat(all_trues, dim=0).numpy() # [Samples, T_out, N, D_out]
    
    results = {}
    horizons = [3, 6, 12]
    for h in horizons:
        # Extract specific step slice (0-indexed, so step 3 is index 2)
        pred_h = preds[:, h - 1, :, :]
        true_h = trues[:, h - 1, :, :]
        mae, rmse, mape = test_eval_metrics(pred_h, true_h)
        results[f"Horizon {h}"] = {"MAE": mae, "RMSE": rmse, "MAPE": mape}
        
    # Average overall horizon performance
    avg_mae, avg_rmse, avg_mape = test_eval_metrics(preds, trues)
    results["Overall"] = {"MAE": avg_mae, "RMSE": avg_rmse, "MAPE": avg_mape}
    
    # Return outputs for visual analysis later
    return results, preds, trues, (torch.cat(all_attn_maps, dim=0) if all_attn_maps else None)

# Run evaluation
baseline_res, baseline_preds, test_trues, _ = evaluate_model_on_test(agcrn_baseline, "AGCRN")
plus_res, plus_preds, _, plus_attn_weights = evaluate_model_on_test(agcrn_plus, "AGCRNPlus")

# Display evaluation table
eval_rows = []
for horizon in ["Horizon 3", "Horizon 6", "Horizon 12", "Overall"]:
    eval_rows.append({
        "Horizon": horizon,
        "Base MAE": baseline_res[horizon]["MAE"],
        "Base RMSE": baseline_res[horizon]["RMSE"],
        "Base MAPE (%)": baseline_res[horizon]["MAPE"] * 100,
        "Plus MAE": plus_res[horizon]["MAE"],
        "Plus RMSE": plus_res[horizon]["RMSE"],
        "Plus MAPE (%)": plus_res[horizon]["MAPE"] * 100,
    })

eval_df = pd.DataFrame(eval_rows)
print("\n=== Comparative Model Performance Summary ===")
print(eval_df.to_string(index=False, formatters={
    "Base MAE": "{:.3f}".format, "Base RMSE": "{:.3f}".format, "Base MAPE (%)": "{:.2f}%".format,
    "Plus MAE": "{:.3f}".format, "Plus RMSE": "{:.3f}".format, "Plus MAPE (%)": "{:.2f}%".format
}))


# Section 9: Research Visualizations

We create publication-quality charts to compare performance, inspect learning dynamics, and analyze internal representations using standard matplotlib.


In [ ]:
# 1. Actual vs Predicted Traffic Curves
plt.figure(figsize=(15, 6))
sample_sensor_idx = 10
steps_to_plot = min(150, len(test_trues))
stop_idx = -len(test_trues) + steps_to_plot
time_index = df.index[-len(test_trues):] if stop_idx == 0 else df.index[-len(test_trues):stop_idx]

plt.plot(time_index, test_trues[:steps_to_plot, 11, sample_sensor_idx, 0], label="Ground Truth", color='black', linewidth=1.5)
plt.plot(time_index, baseline_preds[:steps_to_plot, 11, sample_sensor_idx, 0], label="AGCRN Baseline (60-min horizon)", color='orange', linestyle='--', alpha=0.8)
plt.plot(time_index, plus_preds[:steps_to_plot, 11, sample_sensor_idx, 0], label="AGCRNPlus (60-min horizon)", color='teal', linestyle='-.', alpha=0.8)

plt.title(f"Traffic Speed Forecasting Comparison (Sensor Index: {sample_sensor_idx})", fontsize=14)
plt.xlabel("Time", fontsize=12)
plt.ylabel("Speed (mph)", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2. Loss Curves Comparison
plt.figure(figsize=(10, 5))
epochs_arr = np.arange(1, config.EPOCHS + 1)
plt.plot(epochs_arr, baseline_train_loss, label="Baseline Train Loss", color='orange', marker='o')
plt.plot(epochs_arr, baseline_val_loss, label="Baseline Val Loss", color='red', marker='x')
plt.plot(epochs_arr, plus_train_loss, label="AGCRNPlus Train Loss", color='teal', marker='^')
plt.plot(epochs_arr, plus_val_loss, label="AGCRNPlus Val Loss", color='blue', marker='v')
plt.title("Training and Validation Loss Curves Comparison", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss (MAE)", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 3. Error Distribution plot (using matplotlib hist)
plt.figure(figsize=(10, 5))
base_errors = (baseline_preds - test_trues).flatten()
plus_errors = (plus_preds - test_trues).flatten()
plt.hist(base_errors, bins=100, alpha=0.5, label="AGCRN Baseline Error", color="orange", density=True)
plt.hist(plus_errors, bins=100, alpha=0.5, label="AGCRNPlus Error", color="teal", density=True)
plt.title("Distribution of Prediction Errors (Residuals)", fontsize=14)
plt.xlabel("Error Value (Speed Difference)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 4. Learned Graph Comparison (Before vs After training)
# We extract the trained node embeddings
with torch.no_grad():
    trained_baseline_adj, trained_baseline_emb = agcrn_baseline.node_embeddings.matmul(agcrn_baseline.node_embeddings.T), agcrn_baseline.node_embeddings
    trained_baseline_adj = F.softmax(F.relu(trained_baseline_adj), dim=-1).cpu().numpy()
    
    trained_plus_adj, trained_plus_emb = agcrn_plus.node_embeddings.matmul(agcrn_plus.node_embeddings.T), agcrn_plus.node_embeddings
    trained_plus_adj = F.softmax(F.relu(trained_plus_adj), dim=-1).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im1 = axes[0].imshow(trained_baseline_adj, cmap='viridis')
axes[0].set_title("Trained Adjacency Matrix - AGCRN Baseline", fontsize=12)
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(trained_plus_adj, cmap='viridis')
axes[1].set_title("Trained Adjacency Matrix - AGCRNPlus", fontsize=12)
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# 5. Node Embeddings Projection
emb_baseline_np = trained_baseline_emb.detach().cpu().numpy()
emb_plus_np = trained_plus_emb.detach().cpu().numpy()

# Apply PCA to compress 10D embeddings to 2D
pca = PCA(n_components=2)
base_pca = pca.fit_transform(emb_baseline_np)
plus_pca = pca.fit_transform(emb_plus_np)

# Apply t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=10)
base_tsne = tsne.fit_transform(emb_baseline_np)
plus_tsne = tsne.fit_transform(emb_plus_np)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0, 0].scatter(base_pca[:, 0], base_pca[:, 1], color='orange', alpha=0.7, edgecolors='k')
axes[0, 0].set_title("PCA projection - AGCRN Baseline Embeddings", fontsize=12)
axes[0, 0].grid(True)

axes[0, 1].scatter(plus_pca[:, 0], plus_pca[:, 1], color='teal', alpha=0.7, edgecolors='k')
axes[0, 1].set_title("PCA projection - AGCRNPlus Embeddings", fontsize=12)
axes[0, 1].grid(True)

axes[1, 0].scatter(base_tsne[:, 0], base_tsne[:, 1], color='orange', alpha=0.7, edgecolors='k')
axes[1, 0].set_title("t-SNE projection - AGCRN Baseline Embeddings", fontsize=12)
axes[1, 0].grid(True)

axes[1, 1].scatter(plus_tsne[:, 0], plus_tsne[:, 1], color='teal', alpha=0.7, edgecolors='k')
axes[1, 1].set_title("t-SNE projection - AGCRNPlus Embeddings", fontsize=12)
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# 6. Attention Heatmaps (AGCRNPlus)
if plus_attn_weights is not None:
    # plus_attn_weights shape: [Batch * Nodes, T_in, T_in]
    avg_attn_map = torch.mean(plus_attn_weights, dim=0).numpy() # [T_in, T_in]
    
    plt.figure(figsize=(8, 7))
    im = plt.imshow(avg_attn_map, cmap="Blues")
    plt.colorbar(im)
    plt.xticks(np.arange(12), np.arange(1, 13))
    plt.yticks(np.arange(12), np.arange(1, 13))
    plt.title("Average Temporal Self-Attention Map in AGCRNPlus", fontsize=14)
    plt.xlabel("Key Timestep (Past 1 Hour)", fontsize=12)
    plt.ylabel("Query Timestep (Past 1 Hour)", fontsize=12)
    plt.tight_layout()
    plt.show()


# Section 10: Research Analysis & Discussion

Based on the implementation, configuration, and comparative experiments conducted on the METR-LA dataset, we analyze the strengths, weaknesses, and potential developments of AGCRN and the enhanced AGCRNPlus architecture.

## 1. Strengths of AGCRN
- **Data-Adaptive Adjacency Matrix**: By using a learnable node embedding dictionary ($E_G$), the model automatically learns spatial relationships directly from the traffic patterns rather than requiring a hardcoded geometric distance graph. This is highly beneficial in scenarios where sensor distance does not accurately reflect connectivity (e.g., one-way highways, flyovers, grid locks).
- **Parameter Efficiency via Weight Pooling**: Creating separate parameters for each node in a graph of size $N$ usually results in $O(N)$ parameter growth. AGCRN's node-adaptive parameter learning uses a shared parameter pool ($W_{pool}$), scaling parameter complexity to $O(1)$ with respect to graph size, preventing overfitting and saving memory.
- **Chebyshev Polynomial Graph Convolution**: Chebyshev polynomial convolutions enable multi-hop local convolution operations, allowing efficient aggregation of neighborhood states without requiring manual graph Laplacian calculation.

## 2. Weaknesses of AGCRN
- **Recurrent State Bottle Neck**: AGCRN models temporal sequence dynamics using a sequential graph recurrent cell structure (AGCRN Cell). Recurrent networks naturally suffer from forgetting long-range dependencies, making 60-minute predictions ($T=12$) heavily dependent on only the most recent time steps.
- **Static Spatial Relationships**: The node embeddings $E_G$ are parameterized as stationary parameters. While they adapt to long-term patterns through backpropagation, the resulting adaptive adjacency matrix $A_{adp}$ is fixed during inference and cannot capture dynamic, short-term spatial dependency changes (such as sudden traffic congestion or accidents).
- **Unidirectional Temporal Modeling**: By selecting only the final state representation from the encoder sequence, the model risks losing intermediate details that might be vital for long-horizon multi-step forecasting.

## 3. Improvements Offered by AGCRNPlus
- **Attention-Augmented Long-Term Temporal Modeling**: By placing a multi-head temporal self-attention block after the spatial recurrent encoder, AGCRNPlus allows the model to attend to all sequence states globally. This resolves the recurrent vanishing gradient/forgetting problem, allowing the network to capture complex, multi-scale temporal dependencies.
- **Order-Aware Sequence Features**: The inclusion of learnable temporal positional embeddings ($E_{pos}$) adds sequence step awareness into the representations. This is highly effective because recurrent sequences don't natively maintain step ordering once passed to permutation-invariant attention maps.
- **Regularized Skip Connections**: Incorporating residual connections, layer normalization, and dropout (0.3) prevents representations from collapsing and ensures gradient stability, resulting in smoother training and better generalization on unseen testing splits.

## 4. Future Directions
- **Dynamic Graph Generation**: Modulate the node embedding values ($E_{t, G}$) dynamically at each step using the current input features ($X_t$). This would allow the generated spatial structure to adapt to real-time events (e.g., accidents or rain) rather than remain static.
- **Multi-Scale Spatial-Temporal Convolutions**: Replace recurrent cells entirely with spatial-temporal convolution blocks (such as causal dilated convolutions) to accelerate training and inference while improving parallel computing throughput.
